# Demonstração: Pipeline Completo Cellpose → MarkerNet → Watershed

Este notebook mostra a arquitetura **em-memória correta** do pipeline:
- **Opção A (Recomendada)**: Todos os dados em RAM, fluxo contínuo
- Sem I/O de disco intermediário
- Sem uso de `ground_truth` como proxy

## Setup

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2
from matplotlib.patches import Rectangle

# Project imports
from src.data.load.monuseg_dataset import MonusegDataset
from src.pipeline.model_pipeline import ModelPipeline
from src.pipeline.steps.cellpose_step import CellposeStep
from src.pipeline.steps.rgba_step import RGBAStep
from src.pipeline.steps.marker_step import MarkerStep
from src.pipeline.steps.segmentation_step import SegmentationStep
from src.utils.logger import logger

print(f"PyTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

ModuleNotFoundError: No module named 'src'

## 1. Carregar Dataset

In [ ]:
# Carregar dataset de teste
dataset = MonusegDataset(
    dataset_name="monuseg",
    config_key="monuseg_test",
    transform=None
)

print(f"Dataset size: {len(dataset)} amostras")
print(f"Primeira amostra:")
sample = dataset[0]
for k, v in sample.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: {v.dtype} {v.shape}")
    elif isinstance(v, dict):
        print(f"  {k}: dict com chaves {list(v.keys())}")
    else:
        print(f"  {k}: {type(v).__name__}")

## 2. Montar o Pipeline (SEM MarkerNet por enquanto)

Por enquanto, vamos criar um pipeline sem o `MarkerStep` porque não temos um modelo MarkerNet treinado.
Mostraremos dois cenários:

1. **Cenário A**: Pipeline sem MarkerNet (apenas Cellpose + Watershed)
2. **Cenário B**: Com MarkerNet (quando disponível)

In [ ]:
# ========================================
# CENÁRIO A: Sem MarkerNet
# ========================================

# Opção 1: Pipeline sem MarkerStep (apenas Cellpose + RGBAStep)
# O MarkerStep com model=None vai usar a segmentação como fallback

pipeline_no_marker = ModelPipeline([
    CellposeStep(batch_size=1),
    RGBAStep(),
    MarkerStep(model=None),  # model=None → usa segmentação como markers
    SegmentationStep(use_distance_map=True),
])

print("Pipeline criado (Cenário A: Sem MarkerNet)")
print(f"Steps: {[s.name for s in pipeline_no_marker.steps]}")

## 3. Executar Pipeline em uma Amostra

In [ ]:
# Pegar uma amostra
sample = dataset[0]

print(f"Processando amostra: {sample['id']}")
print(f"Forma da imagem: {sample['image'].shape}")
print(f"Ground truth disponível: {sample.get('ground_truth', 'Não').shape if 'ground_truth' in sample else 'Não'}")

# Preparar input do pipeline (apenas imagem)
pipeline_input = {
    'image': sample['image'],
    'id': sample['id']
}

# Executar pipeline (em-memória)
result = pipeline_no_marker.forward(pipeline_input, verbose=True)

print(f"\nPipeline completo! Chaves no resultado:")
for k, v in result.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: {v.dtype} {v.shape}")
    elif isinstance(v, str):
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {type(v).__name__}")

## 4. Visualizar Resultados em Cada Etapa

In [ ]:
# Visualizar o fluxo através do pipeline
sample = dataset[0]

# Rodar cada passo manualmente para capturar intermediários
pipeline_input = {'image': sample['image'], 'id': sample['id']}

# Passo 1: Cellpose
data = pipeline_no_marker.steps[0](pipeline_input)
cellpose_seg = data['segmentation'].copy()

# Passo 2: RGBA
data = pipeline_no_marker.steps[1](data)
rgba = data['rgba'].copy()

# Passo 3: MarkerStep
data = pipeline_no_marker.steps[2](data)
markers = data['markers'].copy()

# Passo 4: SegmentationStep
data = pipeline_no_marker.steps[3](data)
final_seg = data['segmentation'].copy()

print(f"Shapes capturados:")
print(f"  Image original: {sample['image'].shape}")
print(f"  Cellpose segmentation: {cellpose_seg.shape}, dtype={cellpose_seg.dtype}, unique labels={np.unique(cellpose_seg)[:10]}...")
print(f"  RGBA: {rgba.shape}")
print(f"  Markers: {markers.shape}, unique values={np.unique(markers)}")
print(f"  Final segmentation: {final_seg.shape}, dtype={final_seg.dtype}, unique labels={np.unique(final_seg)[:10]}...")

In [ ]:
# Visualizar
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Normalizações para visualização
def normalize_for_display(img):
    if img.dtype == np.uint8:
        return img.astype(np.float32) / 255.0
    elif img.max() > 0:
        return (img - img.min()) / (img.max() - img.min())
    return img

# Primeira linha: Imagens originais
axes[0, 0].imshow(normalize_for_display(sample['image']), cmap='gray')
axes[0, 0].set_title("1. Imagem Original (Monuseg)")
axes[0, 0].axis('off')

axes[0, 1].imshow(cellpose_seg, cmap='nipy_spectral')
axes[0, 1].set_title(f"2. Cellpose Segmentation ({np.max(cellpose_seg)} células)")
axes[0, 1].axis('off')

axes[0, 2].imshow(rgba)
axes[0, 2].set_title("3. RGBA (RGB + alpha channel)")
axes[0, 2].axis('off')

# Segunda linha: Markers e resultado final
axes[1, 0].imshow(markers, cmap='gray')
axes[1, 0].set_title("4. Markers (Binarizado)")
axes[1, 0].axis('off')

axes[1, 1].imshow(final_seg, cmap='nipy_spectral')
axes[1, 1].set_title(f"5. Final Segmentation ({np.max(final_seg)} células)")
axes[1, 1].axis('off')

# Terceira coluna: Ground truth (para comparação)
if 'ground_truth' in sample:
    axes[1, 2].imshow(sample['ground_truth'], cmap='gray')
    axes[1, 2].set_title("Ground Truth (Referência)")
else:
    axes[1, 2].text(0.5, 0.5, "Ground Truth\nNão disponível", 
                    ha='center', va='center', fontsize=12)
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig('pipeline_flow.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nVisualizações geradas com sucesso!")

## 5. QUANDO MarkerNet Estiver Disponível

Quando você treinar o MarkerNet, use assim:

In [ ]:
# ========================================
# CENÁRIO B: Com MarkerNet Treinado
# ========================================

# from src.models.networks.marker_net import MarkerNet

# # 1. Carregar modelo treinado
# config = {
#     "encoder_name": "resnet34",
#     "pretrained": False,
#     "in_channels": 4,
#     "threshold": 0.5
# }
# marker_net = MarkerNet(config)
# marker_net.load("path/to/marker_net_checkpoint.pt")

# # 2. Criar pipeline COM MarkerNet
# pipeline_with_marker = ModelPipeline([
#     CellposeStep(batch_size=1),
#     RGBAStep(),
#     MarkerStep(model=marker_net, target_size=256, threshold=0.5),
#     SegmentationStep(use_distance_map=True),
# ])

# # 3. Usar exatamente como antes
# pipeline_input = {'image': sample['image'], 'id': sample['id']}
# result = pipeline_with_marker.forward(pipeline_input, verbose=True)

# # result terá:
# # - result['segmentation']: segmentação refinada
# # - result['markers']: markers gerados pelo MarkerNet
# # - result['rgba']: RGBA intermediário
# # - result['flows']: flows do Cellpose
# # etc.

print("Exemplo de código para quando MarkerNet estiver pronto!")

## 6. Para Batch Processing (DataLoader)

Como integrar no seu loop de treinamento:

In [ ]:
# ========================================
# Integração com DataLoader e Training
# ========================================

from torch.utils.data import DataLoader

# Dataset (SEM transformações - deixa para o pipeline)
dataset = MonusegDataset(
    dataset_name="monuseg",
    config_key="monuseg_test",
    transform=None
)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0  # Processamento sequencial por simplicidade
)

# Pipeline
pipeline = ModelPipeline([
    CellposeStep(batch_size=4),  # Cellpose pode processar batch
    RGBAStep(),
    MarkerStep(model=None),
    SegmentationStep(),
])

print(f"DataLoader com {len(dataloader)} batches")
print(f"\nExemplo de loop de processamento:")
print("""
for batch_idx, batch in enumerate(dataloader):
    # batch contém:
    # - batch['image']: (B, H, W) ou (B, H, W, C)
    # - batch['id']: List[str]
    # - batch['ground_truth']: (B, H, W) [OPCIONAL - apenas para loss]
    # - batch['meta']: Dict com metadados
    
    # Processar cada imagem através do pipeline
    results = []
    for i in range(len(batch['image'])):
        sample_input = {
            'image': batch['image'][i],
            'id': batch['id'][i]
        }
        sample_output = pipeline.forward(sample_input)
        results.append(sample_output)
    
    # Agora você tem:
    # - results[i]['segmentation']: segmentação refinada pelo pipeline
    # - results[i]['markers']: markers
    # - batch['ground_truth'][i]: ground truth para calcular loss
    
    # Calcular loss comparando:
    predicted_seg = results[i]['segmentation']
    gt_mask = batch['ground_truth'][i]
    loss = compute_loss(predicted_seg, gt_mask)
""")

## Resumo da Arquitetura

### ✅ Fluxo Em-Memória (RECOMENDADO)

```
┌─────────────────────────────────────────────────────────────┐
 │                      Imagem (H, W)                          │
 │                    MonusegDataset                           │
 └────────────────────────┬────────────────────────────────────┘
                          │
                          ▼
        ┌─────────────────────────────────────┐
        │      CellposeStep (GPU)             │
        │  Output: segmentation (H, W int)    │
        └────────────┬────────────────────────┘
                     │
                     ▼
        ┌─────────────────────────────────────┐
        │      RGBAStep (CPU)                 │
        │  Output: rgba (H, W, 4)             │
        └────────────┬────────────────────────┘
                     │
                     ▼
        ┌─────────────────────────────────────┐
        │      MarkerStep (GPU/CPU)           │
        │  Input: rgba (256, 256, 4)          │
        │  Output: markers (H, W)             │
        └────────────┬────────────────────────┘
                     │
                     ▼
        ┌─────────────────────────────────────┐
        │    SegmentationStep (CPU)           │
        │  Watershed refinement               │
        │  Output: refined_seg (H, W)         │
        └────────────┬────────────────────────┘
                     │
                     ▼
      ┌───────────────────────────────────┐
      │  Segmentação final refinada       │
      │  Pronta para Loss computation     │
      │  (vs ground_truth)                │
      └───────────────────────────────────┘
```

### ✅ Vantagens
- Sem I/O de disco
- Fluxo contínuo em RAM
- Fácil debugging
- Altamente escalável
- Compatible com DataLoaders

### ✅ Quando Integrar MarkerNet
1. Treinar MarkerNet no seu `experiment_1.ipynb`
2. Salvar checkpoint: `marker_net.save('checkpoint.pt')`
3. Carregar no pipeline: `MarkerStep(model=marker_net)`
4. Pronto! Nenhuma outra mudança necessária